### 2 1 1 종목의 기초 정보 수집하기.md


In [ ]:
!pip install finance-datareader


In [ ]:
!pip install --upgrade finance-datareader  


In [ ]:
# 코스피 종목 정보 수집 
df = fdr.StockListing('KOSPI')

# 종목코드, 종목명, 시가총액만 선택 
df = df[['Code','Name','Marcap']]

# 최초 5건 화면 출력
df.head()


In [ ]:
# 코스닥 종목 정보 수집 
df = fdr.StockListing('KOSDAQ')

# 종목코드, 종목명, 시가총액만 선택 
df = df[['Code','Name','Marcap']]

# 최초 5건 화면 출력
df.head()


In [ ]:
# KOSPI 종목 리스트에서 종목코드(Code), 종목명(Name), 시가총액(Marcap)만 선별
df = fdr.StockListing('KOSPI')[['Code', 'Name', 'Marcap']]

# 우선주, 최근상장 종목, 리츠 제외 (df는 시가총액 상위 순으로 정렬) 
df = df.query("Name not in ['삼성전자우', '현대차2우B', '미래에셋증권2우B','삼성에피스홀딩스','맥쿼리인프라']")

# 시가총액 상위 200개 종목 추출 
df = df.iloc[:200]

# 인덱스 번호 재설정 (0부터 순차 부여)
df = df.reset_index(drop=True) 

# 결과 데이터 최초 5건 확인
df.head()


### 2 1 2 모멘텀 데이터 수집하기.md


In [ ]:
fdr.DataReader('005930', '20250416', '20250418')   # 삼성전자의 25.4.16~25.4.18 주가


In [ ]:
df['Close'].plot(kind='line', grid=True)


In [ ]:
df['Volume'].rolling(window=5).mean()  # 5일 이동평균 계산 


In [ ]:
vol_short = df['Volume'].rolling(window=5).mean().iloc[-1] # 거래량 5일이동평균의 마지막 데이터
vol_long = df['Volume'].rolling(window=20).mean().iloc[-1] # 거래량20일이동평균의 마지막 데이터
vol = ( vol_short / vol_long - 1) * 100 # 거래량 변동성 계산 


### 2 1 3 밸류 데이터 수집하기.md


In [ ]:
# 재무제표 데이터 추출 (삼성전자, 연간 K-IFRS 연결 기준)
data = fdr.SnapDataReader('NAVER/FINSTATE-2Y/005930') 

# 미래 추정치 제외 (영업이익 결측치 삭제) 
data = data.dropna(subset='영업이익')

# PER, PBR 출력 
data[['PER(배)', 'PBR(배)']]


In [ ]:
print(data['PER(배)'].iloc[-1])
> 10.49

print(data['PBR(배)'].iloc[-1])
> 0.89


### 2 1 4 퀄리티 데이터 수집하기.md


In [ ]:
# 대상 컬럼 지정 
cols = ['매출액', '영업이익', '당기순이익', 'ROE(%)'] 

# 재무 데이터 추출 : 분기 K-IFRS 연결 기준 
data = fdr.SnapDataReader('NAVER/FINSTATE-2Q/005930')[cols] 

# 미래 추정치 제외 : 영업이익 결측치 삭제 
data = data.dropna(subset='영업이익')

# 결과 화면 출력 
data


In [ ]:
data[['매출액','영업이익','당기순이익']].pct_change()*100


In [ ]:
(data[['매출액','영업이익','당기순이익']].pct_change()*100).iloc[-1].to_list()
   
> [15.416348, 160.176215, 138.951216]


In [ ]:
data['ROE(%)'].iloc[-1]

> 8.33


### 2 2 1 MultiFactor 라이브러리 이해하기.md


In [ ]:
!pip install MultiFactor


In [ ]:
!pip install --upgrade MultiFactor


In [ ]:
from MultiFactor import MultiFactor  # 라이브러리 불러오기
mf = MultiFactor(N=5)  # 멀티팩터 객체 생성: 시가총액 상위 5개 종목 추출
mf.get_score()         # 멀티팩터 종합 점수 산출


### 2 2 2 종합지표 생성하기.md


In [ ]:
cols = ['scode', 'sname', 'mom_price', 'mom_vol', 'PER', 'PBR', 
        'revenue_rate', 'oper_income_rate', 'net_income_rate', 'ROE']  #  종합지표 원본 컬럼명 정의
df[cols].head()  # 상위 5개 종목 결과 출력


In [ ]:
cols = ['scode', 'sname',  '모멘텀_주가', '모멘텀_거래량', '밸류_PER', '밸류_PBR', 
             '퀄리티_ROE', '퀄리티_매출증가', '퀄리티_영업이익증가', '퀄리티_순이익증가']  #  종합지표 순위 컬럼명 정의
df[cols].head()  # 상위 5개 종목 결과 출력


In [ ]:
cols = ['scode', 'sname', '종합점수', '종합순위', '종합순위_퍼센트']  #  종합점수/순위 컬럼명 정의
df[cols].head()  # 상위 5개 종목 결과 출력


In [ ]:
# 1. 라이브러리 불러오기
from MultiFactor import MultiFactor  

# 2. 멀티팩터 객체 생성: 시가총액 상위 100 종목
mf = MultiFactor(N=100)  

# 3. 멀티팩터 종합점수 산출하여 데이터프레임(df)에 저장 
df = mf.get_score()  

# 4. 분석된 종목들을 종합 점수 순으로 10개(Ngroup=10) 그룹으로 나누어 요약 출력
mf.get_Ngroup(df, Ngroup=10)

# [출력 결과]
# 1 : SK스퀘어, SK하이닉스, 삼성E&A, HD현대, 에이피알, 한국금융지주, 삼성전자, LG이노텍, 키움증권, LS ELECTRIC
# 2 : 효성중공업, NH투자증권, 이수페타시스, 삼성물산, 삼성증권, HD현대일렉트릭, LG씨엔에스, LS, HD한국조선해양, 현대건설
# 3 : 코오롱티슈진, 한화, 고려아연, 하나금융지주, JB금융지주, 미래에셋증권, KB금융, 삼천당제약, 기업은행, 엘앤에프
# 4 : DB손해보험, 기아, 한전기술, 셀트리온, 우리금융지주, 펩트론, BNK금융지주, 대우건설, 한국항공우주, 현대글로비스
# 5 : 삼성생명, 신한지주, HD현대중공업, 한화에어로스페이스, 현대모비스, 한화솔루션, 현대차, 삼양식품, 삼성카드, 에이비엘바이오
# 6 : SK바이오팜, LG유플러스, 한미약품, S-Oil, GS, KT, 한화오션, 삼성화재, 두산, 삼성전기
# 7 : SK텔레콤, 카카오뱅크, 한국전력, 한국타이어앤테크놀로지, HMM, 삼성중공업, 포스코인터내셔널, SK, 현대로템, 카카오페이
# 8 : 대한항공, 현대오토에버, CJ, LIG넥스원, 삼성SDI, HD건설기계, HD현대마린솔루션, 두산에너빌리티, 크래프톤, 하이브
# 9 : HLB, 삼성에스디에스, KT&G, 아모레퍼시픽, 유한양행, 한화시스템, SK이노베이션, LG전자, 삼성에피스홀딩스, NAVER
# 10 : LG, 레인보우로보틱스, 한미반도체, POSCO홀딩스, LG에너지솔루션, 카카오, LG화학, 삼성바이오로직스, 한진칼, 포스코퓨처엠


### 2 3 3  종합지표 최적화하기.md


In [ ]:
# 1. 라이브러리 불러오기
from MultiFactor import MultiFactor  

# 2. 멀티팩터 객체 생성: 시가총액 상위 100 종목
mf = MultiFactor(N=100)  

# 3. 멀티팩터 종합점수 산출하여 데이터프레임(df)에 저장 
df = mf.get_score()  


In [ ]:
# 화면 출력 컬럼 정의
cols = ['scode', 'sname',  '모멘텀_주가', '모멘텀_거래량', '밸류_PER', '밸류_PBR', 
             '퀄리티_ROE', '퀄리티_매출증가', '퀄리티_영업이익증가', '퀄리티_순이익증가',
             '종합점수', '종합순위', '종합순위_퍼센트']  

# 1. 가치 성장 전략 적용 (밸류 + 퀄리티 조합)
data = mf.get_score_adj_weight(df, weight='가치성장') 

# 최상위 5종목 출력
data[cols].head()  


In [ ]:
# 화면 출력 컬럼 정의
cols = ['scode', 'sname',  '모멘텀_주가', '모멘텀_거래량', '밸류_PER', '밸류_PBR', 
             '퀄리티_ROE', '퀄리티_매출증가', '퀄리티_영업이익증가', '퀄리티_순이익증가',
             '종합점수', '종합순위', '종합순위_퍼센트']  

# 2. 추세 성장 전략 적용 (모멘텀 + 퀄리티 조합)
data = mf.get_score_adj_weight(df, weight='추세성장') 

# 최상위 5종목 출력
data[cols].head()  


In [ ]:
# 화면 출력 컬럼 정의
cols = ['scode', 'sname',  '모멘텀_주가', '모멘텀_거래량', '밸류_PER', '밸류_PBR', 
             '퀄리티_ROE', '퀄리티_매출증가', '퀄리티_영업이익증가', '퀄리티_순이익증가',
             '종합점수', '종합순위', '종합순위_퍼센트']  

# 3. 역발상 전략 적용 (밸류+모멘텀 조합) 
data = mf.get_score_adj_weight(df, weight='역발상') 

# 최상위 5종목 출력
data[cols].head()  
